# LRO-like orbit around the Moon
Philadelphia Space Forum · Scott Blender

Participant edition — fill the five `None` values, then run all cells.

Follows the live-demo workflow on slides 24–25 of *Cislunar Space Fundamentals*: constants → two-body dynamics → orbital elements → propagation → animation.

**Setup:** Python 3.11+; run `python -m pip install -r requirements.txt` in this folder, then `python -m jupyterlab`. Select a Python kernel. The animation uses embedded JavaScript controls; no FFmpeg, external imagery, ephemeris download, or internet is needed after installation. Trust the notebook if Jupyter hides the animation.

## 1. Constants and physical parameters
Use **kilometers, seconds, and degrees** at input; convert angles internally to radians. The Moon is a spherical point-mass gravity source.

### Five inputs to complete
| Input | Meaning | Hint |
|---|---|---|
| 1. `MU_MOON` | Gravity strength, GM | Use JPL's lunar value in km³/s², not G alone. |
| 2. `altitude_km` | Orbit size above the mean radius | Use the original LRO mapping altitude from NASA. |
| 3. `eccentricity` | Shape | Choose a small, nonnegative value near zero. |
| 4. `inclination_deg` | Orbital-plane tilt | Choose a polar orbit. |
| 5. `number_of_periods` | Simulation length | Choose a few complete revolutions. |

Only these five values are missing. The complete edition provides one worked set.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

MU_MOON = None  # INPUT 1: km^3/s^2; gravitational parameter, GM
R_MOON = 1737.4       # km; mean spherical radius

## 2. Two-body propagation with `solve_ivp`
The state is $[x,y,z,v_x,v_y,v_z]$ in a Moon-centered inertial (MCI) frame:
$$\dot{\mathbf r}=\mathbf v,\qquad \dot{\mathbf v}=-\mu\frac{\mathbf r}{\|\mathbf r\|^3}.$$
`t_eval` sets output times; the integrator chooses its internal steps. DOP853 is an adaptive Runge–Kutta method.

In [ ]:
def two_body_rhs(t, state, mu):
    r = state[:3]
    v = state[3:]
    acceleration = -mu * r / np.linalg.norm(r)**3
    return np.concatenate((v, acceleration))


def propagate_two_body(state0, duration_s, mu, samples=1201):
    times = np.linspace(0.0, duration_s, samples)
    solution = solve_ivp(
        two_body_rhs, (0.0, duration_s), state0,
        args=(mu,), t_eval=times, method="DOP853",
        rtol=1e-10, atol=1e-12, max_step=duration_s / 1000,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    return solution.t, solution.y.T

## 3. LRO-like orbit and orbital elements
Use an **illustrative** near-circular, low-altitude polar orbit inspired by LRO's original ~50 km mapping orbit, not a current or reconstructed LRO trajectory. Earth/Sun gravity, lunar gravity harmonics, maneuvers and lunar rotation are omitted.

The six elements are $a,e,i,\Omega,\omega,\nu_0$. Here $a=R_{Moon}+h$; for $e>0$, $h$ is the altitude corresponding to the semi-major axis, not constant altitude. Angles use an idealized MCI frame with its reference plane at the lunar equator; the x-axis is arbitrary and fixed. RAAN and periapsis orientation are teaching choices, not measured LRO values.

The slides use $M_0$ (mean anomaly). We use $\nu_0$ (true anomaly) to construct the state directly; both are zero at our chosen initial periapsis. They differ in general. For a circular orbit, periapsis is undefined; its angle simply fixes our phase convention.

In [ ]:
def elements_to_state(a, e, i_deg, raan_deg, argp_deg, nu_deg, mu):
    i, raan, argp, nu = np.deg2rad([i_deg, raan_deg, argp_deg, nu_deg])
    p = a * (1.0 - e**2)
    r_pqw = p / (1.0 + e * np.cos(nu)) * np.array([np.cos(nu), np.sin(nu), 0.0])
    v_pqw = np.sqrt(mu / p) * np.array([-np.sin(nu), e + np.cos(nu), 0.0])

    def R3(angle):
        c, s = np.cos(angle), np.sin(angle)
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

    c, s = np.cos(i), np.sin(i)
    R1 = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    rotation = R3(raan) @ R1 @ R3(argp)
    return np.concatenate((rotation @ r_pqw, rotation @ v_pqw))

In [ ]:
altitude_km = None       # INPUT 2: sets a = lunar radius + altitude
eccentricity = None      # INPUT 3: orbit shape; zero is circular
inclination_deg = None    # INPUT 4: angle to the reference equator
raan_deg = 30.0
arg_periapsis_deg = 0.0
true_anomaly_deg = 0.0

if any(value is None for value in (MU_MOON, altitude_km, eccentricity, inclination_deg)):
    raise ValueError("Fill inputs 1–4, then rerun the notebook from the top.")
if not (MU_MOON > 0 and altitude_km > 0 and 0 <= eccentricity < 1 and 0 <= inclination_deg <= 180):
    raise ValueError("Use positive GM/altitude, 0 <= e < 1, and 0 <= inclination <= 180 degrees.")

a_km = R_MOON + altitude_km
periapsis_altitude_km = a_km * (1 - eccentricity) - R_MOON
apoapsis_altitude_km = a_km * (1 + eccentricity) - R_MOON
if periapsis_altitude_km <= 0:
    raise ValueError("This orbit intersects the spherical Moon: lower e or increase altitude.")

state0 = elements_to_state(a_km, eccentricity, inclination_deg, raan_deg,
                           arg_periapsis_deg, true_anomaly_deg, MU_MOON)
period_s = 2 * np.pi * np.sqrt(a_km**3 / MU_MOON)
print(f"a = {a_km:.1f} km; e = {eccentricity}; i = {inclination_deg:.1f} deg")
print(f"Periapsis / apoapsis altitude: {periapsis_altitude_km:.2f} / {apoapsis_altitude_km:.2f} km")
print(f"Period: {period_s / 60:.2f} min; initial speed: {np.linalg.norm(state0[3:]):.3f} km/s")

## 4. Propagate for multiple periods
Kepler's period is $T=2\pi\sqrt{a^3/\mu}$. `states` contains position [km] and velocity [km/s] at each output time.

In [ ]:
number_of_periods = None  # INPUT 5: how long to propagate
if number_of_periods is None or not isinstance(number_of_periods, (int, float)) or not np.isfinite(number_of_periods) or number_of_periods <= 0:
    raise ValueError("Fill input 5 with a positive number of orbital periods.")

duration_s = number_of_periods * period_s
time_s, states = propagate_two_body(state0, duration_s, MU_MOON)
position_km = states[:, :3]
velocity_km_s = states[:, 3:]
print(f"Propagated {number_of_periods:g} periods: {duration_s / 3600:.2f} hours, {len(time_s)} output samples.")

### Quick numerical check
Two-body specific energy and angular-momentum magnitude should stay constant, within numerical error.

In [ ]:
radius = np.linalg.norm(position_km, axis=1)
energy = np.sum(velocity_km_s**2, axis=1) / 2 - MU_MOON / radius
momentum = np.linalg.norm(np.cross(position_km, velocity_km_s), axis=1)
energy_error = np.max(np.abs((energy - energy[0]) / energy[0]))
momentum_error = np.max(np.abs((momentum - momentum[0]) / momentum[0]))
print(f"Maximum relative energy / angular-momentum drift: {energy_error:.2e} / {momentum_error:.2e}")
assert energy_error < 1e-7 and momentum_error < 1e-7
assert np.all(radius > R_MOON)

## 5. Animate the trajectory
The cyan curve is the orbit; the orange point is the spacecraft. The sphere and equal axis scales show the Moon's physical size. This is an inertial trajectory view, not a rotating ground track. Repeated periods retrace the same ellipse in this ideal model. Playback is accelerated; the elapsed-time label shows simulation time.

In [ ]:
def animate_orbit(time_s, position_km, moon_radius_km, frames=240, fps=30):
    indices = np.unique(np.linspace(0, len(time_s) - 1, frames).astype(int))
    fig = plt.figure(figsize=(6, 6), dpi=85, facecolor="#090909")
    ax = fig.add_subplot(111, projection="3d", facecolor="#090909")
    ax.set_proj_type("ortho")
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 48), np.linspace(0, np.pi, 24))
    ax.plot_surface(moon_radius_km*np.cos(u)*np.sin(v),
                    moon_radius_km*np.sin(u)*np.sin(v),
                    moon_radius_km*np.cos(v), color="#aaaaaa", alpha=0.45,
                    linewidth=0, shade=True)
    ax.plot(*position_km.T, color="#68C4FF", alpha=0.3, lw=1)
    trail, = ax.plot([], [], [], color="#00B5EC", lw=2)
    satellite, = ax.plot([], [], [], "o", color="#FFB454", ms=6)
    label = ax.text2D(0.04, 0.94, "", transform=ax.transAxes, color="white")
    limit = 1.15 * max(moon_radius_km, np.linalg.norm(position_km, axis=1).max())
    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit),
           xlabel="MCI x [km]", ylabel="MCI y [km]", zlabel="MCI z [km]")
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=20, azim=-55)
    ax.set_title("LRO-like orbit · Moon-centered inertial frame", color="white", pad=18)
    ax.tick_params(colors="white", labelsize=7)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.label.set_color("white")
        axis.set_pane_color((0.04, 0.04, 0.04, 1))
    fig.subplots_adjust(left=0.02, right=0.92, bottom=0.06, top=0.9)

    def update(frame):
        index = indices[frame]
        trail.set_data_3d(*position_km[:index+1].T)
        satellite.set_data_3d(*position_km[index:index+1].T)
        label.set_text(f"Elapsed time: {time_s[index] / 3600:.2f} h")
        return trail, satellite, label

    animation = FuncAnimation(fig, update, frames=len(indices), interval=1000/fps,
                              blit=False, repeat=True)
    plt.close(fig)
    return animation

In [ ]:
animation = animate_orbit(time_s, position_km, R_MOON)
with plt.rc_context({"animation.embed_limit": 80}):
    display(HTML(animation.to_jshtml(fps=30)))

## Try changing one input
Increase altitude: how does the period change? Increase eccentricity while keeping periapsis above the surface: where is motion faster? Change inclination: how does the orbital plane move?

## Sources
- Workshop: Scott Blender, *Cislunar Space Fundamentals*, slides 11–19 and 24–25.
- [JPL astrodynamic parameters](https://ssd.jpl.nasa.gov/astro_par.html): lunar GM.
- [JPL satellite physical parameters](https://ssd.jpl.nasa.gov/sats/phys_par/): mean lunar radius.
- [NASA LRO mission overview](https://science.nasa.gov/mission/lro/about/): original ~50 km mapping orbit.
- [SciPy solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).